# Which professors grade above or below their colleagues? (UBCGrades)

1. Pulls every graded section for a campus (all subjects) from the [UBCGrades API](https://ubcgrades.com/api-reference).
2. Keeps only professors who have **taught a lot** (thresholds in the settings cell).
3. For each one, compares their average in every course they taught with the enrollment-weighted average of **everyone else who taught that same course**.
4. Shows the results in an **interactive dashboard** (search, filters, sortable columns, click a professor for a per-course breakdown) and also saves a CSV and a standalone HTML file.

Run the cells top to bottom. The first load can take a few minutes; it is cached to disk afterwards.

In [ ]:
%pip install requests pandas numpy

In [ ]:
# ---- SETTINGS: edit these ----
CAMPUS = "UBCO"            # "UBCO" or "UBCV"
SUBJECTS = "ALL"           # "ALL", or a comma-separated list like "PHYS,MATH,ASTR"
CSV_DIR = None             # e.g. r"C:\Users\gdoer\Downloads\grades" to read CSV files instead of the API

# Who counts as someone who "taught a lot":
MIN_SECTIONS = 7          # at least this many graded sections taught...
MIN_TERMS = 4              # ...spread over at least this many different terms (e.g. 2021W, 2022S)...
MIN_COURSES = 2            # ...and at least this many courses where other instructors also taught
MIN_OTHER_SECTIONS = 2     # a course only counts if others taught it in at least this many sections

SORT_BY_ABS = False        # False: most above-others first. True: biggest gap in either direction first
CACHE_FILE = "grades_cache.pkl"   # speeds up reruns; set to None to disable
REFRESH = False            # True = ignore the cache and re-download
OUT_CSV = "prof_comparison.csv"
OUT_HTML = "index.html"             # the web page GitHub Pages serves
DASH_HEIGHT = 820                   # height (px) of the dashboard shown inside the notebook
DEBUG = False

In [ ]:
# ---- Imports and data access ----
import csv, glob, os, pickle, re
import statistics as st
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from datetime import date

import numpy as np
import pandas as pd
import requests

BASE = "https://ubcgrades.com/api"
VERSIONS = ["v3", "v2"]   # v3 = recent terms, v2 = older terms

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (grade-compare-script)",
                        "Accept": "application/json"})
stats = {"ok": 0, "fail": 0, "last_error": None}


def get(version, path):
    url = f"{BASE}/{version}{path}"
    try:
        r = session.get(url, timeout=30)
        if r.status_code != 200:
            stats["fail"] += 1
            stats["last_error"] = f"{r.status_code} for {url}"
            return None
        stats["ok"] += 1
        return r.json()
    except (requests.RequestException, ValueError) as e:
        stats["fail"] += 1
        stats["last_error"] = f"{type(e).__name__} for {url}"
        return None


def norm(rec):
    """Lowercase keys and strip punctuation: 'Std dev' -> 'stddev'."""
    return {re.sub(r"[^a-z0-9]", "", str(k).lower()): v for k, v in rec.items()}


def as_records(data):
    if data is None:
        return []
    if isinstance(data, list):
        return [norm(d) for d in data if isinstance(d, dict)]
    if isinstance(data, dict):
        for v in data.values():
            if isinstance(v, list) and v and isinstance(v[0], dict):
                return [norm(d) for d in v]
        return [norm(data)]
    return []


def first(r, keys):
    for k in keys:
        if r.get(k) not in (None, ""):
            return r[k]
    return None


def extract_codes(data, campus):
    """Collect subject codes (2-4 capital letters) from whatever shape the API returns."""
    found = set()

    def walk(x):
        if isinstance(x, str):
            if re.fullmatch(r"[A-Z]{2,4}", x):
                found.add(x)
        elif isinstance(x, dict):
            for v in x.values():
                walk(v)
        elif isinstance(x, list):
            for v in x:
                walk(v)

    walk(data)
    found.discard(campus.upper())
    return found


def get_terms(campus):
    found = set()
    for v in VERSIONS:
        data = get(v, f"/yearsessions/{campus}")
        found.update(re.findall(r"\b\d{4}[WS]\b", str(data)))
    if found:
        return sorted(found)
    return [f"{y}{s}" for y in range(2008, date.today().year + 1) for s in "SW"]


def get_subjects(campus, ys):
    for v in VERSIONS:
        codes = extract_codes(get(v, f"/subjects/{campus}/{ys}"), campus)
        if codes:
            return sorted(codes)
    return []


def fetch_task(task):
    campus, ys, subject = task
    for v in VERSIONS:
        recs = as_records(get(v, f"/grades/{campus}/{ys}/{subject}"))
        if recs:
            for r in recs:
                r.setdefault("subject", subject)
                r["term"] = ys
            return recs
    return []


def records_from_api(campus, subjects):
    terms = get_terms(campus)
    print(f"Found {len(terms)} terms. Looking up subjects...")
    tasks = []
    if "ALL" in subjects:
        with ThreadPoolExecutor(12) as ex:
            for ys, subs in zip(terms, ex.map(lambda t: get_subjects(campus, t), terms)):
                tasks += [(campus, ys, s) for s in subs]
    else:
        tasks = [(campus, ys, s) for ys in terms for s in subjects]
    print(f"Downloading {len(tasks)} term/subject pages (this is the slow part)...")
    with ThreadPoolExecutor(12) as ex:
        for i, recs in enumerate(ex.map(fetch_task, tasks), 1):
            if i % 100 == 0:
                print(f"  {i}/{len(tasks)}")
            for r in recs:
                yield r
    print(f"API requests: {stats['ok']} ok, {stats['fail']} failed "
          f"(failures are normal for term/subject pairs with no data).")
    if stats["ok"] == 0:
        print("Every request failed. Last error:", stats["last_error"])


def records_from_csv(csv_dir, campus, subjects):
    files = glob.glob(os.path.join(csv_dir, "**", "*.csv"), recursive=True)
    print(f"Reading {len(files)} CSV files from {csv_dir}...")
    for f in files:
        with open(f, newline="", encoding="utf-8-sig") as fh:
            for row in csv.DictReader(fh):
                r = norm(row)
                if r.get("campus", campus).upper() != campus.upper():
                    continue
                if "ALL" not in subjects and r.get("subject", "").upper() not in subjects:
                    continue
                r["term"] = f"{r.get('year', '')}{r.get('session', '')}"
                yield r


def load_data(campus, subjects, csv_dir=None, cache_file=None, refresh=False, debug=False):
    """Return a compact list of graded sections: key=(subject, course), avg, enr, who=[names], term."""
    stats.update(ok=0, fail=0, last_error=None)
    subs = [x.upper() for x in re.split(r"[,\s]+", subjects) if x]
    meta = {"campus": campus, "subjects": subs}

    if cache_file and not csv_dir and not refresh and os.path.exists(cache_file):
        try:
            with open(cache_file, "rb") as fh:
                blob = pickle.load(fh)
            if blob.get("meta") == meta:
                print(f"Loaded {len(blob['sections'])} sections from cache ({cache_file}).")
                return blob["sections"]
        except Exception:
            pass

    if debug:
        dbg = subs[0] if subs[0] != "ALL" else "PHYS"
        data = get("v3", f"/grades/{campus}/2021W/{dbg}")
        print("Sample response:", type(data).__name__, str(data)[:500])
        print("Last error:", stats["last_error"])

    source = records_from_csv(csv_dir, campus, subs) if csv_dir else records_from_api(campus, subs)
    sections = []
    for r in source:
        if str(r.get("section", "")).upper() == "OVERALL":
            continue
        try:
            avg = float(first(r, ["avg", "average"]))
            enr = float(first(r, ["enrolled", "reported", "enrollment"]))
        except (TypeError, ValueError):
            continue
        if enr <= 0:
            continue
        who = str(first(r, ["educators", "professor", "instructor"]) or "")
        names = sorted({" ".join(n.split()) for n in who.split(";") if n.strip()})
        sections.append({
            "key": (str(r.get("subject", "?")).upper(), str(r.get("course", "?"))),
            "avg": avg, "enr": enr, "who": names, "term": str(r.get("term", "")),
        })
    print(f"Loaded {len(sections)} graded sections.")

    if cache_file and not csv_dir and sections:
        with open(cache_file, "wb") as fh:
            pickle.dump({"meta": meta, "sections": sections}, fh)
        print(f"Cached to {cache_file}.")
    return sections

In [ ]:
# ---- Professor selection and comparison ----
def select_profs(sections, min_sections, min_terms):
    """Names that appear in enough sections and terms to count as having 'taught a lot'."""
    n_sec, terms = Counter(), defaultdict(set)
    for s in sections:
        for p in s["who"]:
            n_sec[p] += 1
            terms[p].add(s["term"])
    keep = {p for p in n_sec if n_sec[p] >= min_sections and len(terms[p]) >= min_terms}
    return keep, n_sec, terms


def compare_all(sections, profs, min_courses=3, min_other_sections=2, boot=2000, details=None):
    """For each prof: their average in each course vs everyone else's average in that same course."""
    tot = defaultdict(lambda: [0.0, 0.0, 0])                       # course -> [sum(avg*enr), sum(enr), n]
    by_prof = defaultdict(lambda: defaultdict(lambda: [0.0, 0.0, 0]))
    terms, subj = defaultdict(set), defaultdict(Counter)
    n_sec = Counter()

    for s in sections:
        a, e, key = s["avg"], s["enr"], s["key"]
        t = tot[key]
        t[0] += a * e; t[1] += e; t[2] += 1
        for p in s["who"]:
            if p in profs:
                c = by_prof[p][key]
                c[0] += a * e; c[1] += e; c[2] += 1
                terms[p].add(s["term"]); subj[p][key[0]] += 1; n_sec[p] += 1

    rng = np.random.default_rng(0)
    rows = []
    for p in profs:
        D, W, P, O, CR = [], [], [], [], []
        for key, c in by_prof[p].items():
            t = tot[key]
            o_e, o_n = t[1] - c[1], t[2] - c[2]
            if o_n < min_other_sections or o_e <= 0:
                continue
            p_avg, o_avg = c[0] / c[1], (t[0] - c[0]) / o_e
            D.append(p_avg - o_avg); W.append(c[1]); P.append(p_avg); O.append(o_avg)
            CR.append({"course": f"{key[0]} {key[1]}", "sections": c[2], "others_sections": o_n,
                       "students": int(c[1]), "prof": round(p_avg, 2), "others": round(o_avg, 2),
                       "diff": round(p_avg - o_avg, 2)})
        if len(D) < min_courses:
            continue
        if details is not None:          # per-course rows for the dashboard
            details[p] = sorted(CR, key=lambda x: -x["students"])
        D, W, P, O = map(np.array, (D, W, P, O))
        diff = float((W * D).sum() / W.sum())
        idx = rng.integers(0, len(D), size=(boot, len(D)))
        bm = (W[idx] * D[idx]).sum(1) / W[idx].sum(1)
        lo, hi = np.percentile(bm, [2.5, 97.5])
        rows.append({
            "Professor": p,
            "Main subjects": ", ".join(s for s, _ in subj[p].most_common(2)),
            "Sections": n_sec[p],
            "Terms": len(terms[p]),
            "Courses compared": len(D),
            "Students": int(W.sum()),
            "Prof avg": float((W * P).sum() / W.sum()),
            "Others avg": float((W * O).sum() / W.sum()),
            "Diff": diff,
            "Median course diff": float(np.median(D)),
            "% courses above": 100 * float((D > 0).mean()),
            "CI low": float(lo),
            "CI high": float(hi),
            "Clear gap?": "yes" if lo > 0 or hi < 0 else "",
        })
    return pd.DataFrame(rows)

In [ ]:
# ---- Interactive dashboard (search, filters, sortable table) ----
import html as _html
import json
from IPython.display import HTML, display

DASHBOARD_TEMPLATE = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>UBC grade comparison</title>
<style>
:root{--bg:#f5f6f8;--panel:#fff;--text:#1c2330;--muted:#6a7485;--line:#e3e7ee;--accent:#2f5bea;--pos:#168a4a;--neg:#c93b3b;--posbg:#e3f4ea;--negbg:#fbe8e8;--hover:#eff3ff;--mark:#fff1a8}
@media (prefers-color-scheme:dark){:root{--bg:#12151b;--panel:#1b2029;--text:#e6e9ef;--muted:#8b95a7;--line:#2a313d;--accent:#7b9bff;--pos:#4cc27f;--neg:#ef7676;--posbg:#183626;--negbg:#3b1d1d;--hover:#222a3a;--mark:#5c4b00}}
*{box-sizing:border-box}
html,body{height:100%;margin:0}
body{background:var(--bg);color:var(--text);font:14px/1.4 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
.app{display:flex;flex-direction:column;height:100%;padding:14px 16px 10px;gap:10px}
h1{font-size:17px;margin:0;font-weight:650}
.sub{color:var(--muted);font-size:12.5px;margin-top:2px}
.toolbar{display:flex;flex-wrap:wrap;gap:8px;align-items:center}
.search{position:relative;flex:1 1 260px;min-width:200px}
.search input{width:100%;padding:8px 30px 8px 32px;border:1px solid var(--line);border-radius:8px;background:var(--panel);color:var(--text);font-size:14px}
.search input:focus,select:focus,.num:focus{outline:2px solid var(--accent);outline-offset:-1px}
.search svg{position:absolute;left:10px;top:50%;transform:translateY(-50%);width:15px;height:15px;color:var(--muted)}
.search .x{position:absolute;right:6px;top:50%;transform:translateY(-50%);border:0;background:none;color:var(--muted);cursor:pointer;font-size:16px;display:none}
select,.num,.btn{padding:7px 9px;border:1px solid var(--line);border-radius:8px;background:var(--panel);color:var(--text);font-size:13px}
.num{width:68px}
.btn{cursor:pointer}
.btn:hover{border-color:var(--accent)}
.seg{display:inline-flex;border:1px solid var(--line);border-radius:8px;overflow:hidden}
.seg button{border:0;background:var(--panel);color:var(--text);padding:7px 11px;cursor:pointer;font-size:13px;border-right:1px solid var(--line)}
.seg button:last-child{border-right:0}
.seg button.on{background:var(--accent);color:#fff}
label.chk{display:inline-flex;align-items:center;gap:5px;font-size:13px;cursor:pointer;user-select:none}
.lab{font-size:12px;color:var(--muted)}
.status{font-size:12.5px;color:var(--muted);display:flex;gap:14px;flex-wrap:wrap}
.status b{color:var(--text)}
.info{display:none;background:var(--panel);border:1px solid var(--line);border-radius:8px;padding:10px 14px;font-size:12.5px;max-height:34vh;overflow:auto}
.info.show{display:block}
.info ul{margin:4px 0 8px 18px;padding:0}
.tablebox{flex:1;overflow:auto;background:var(--panel);border:1px solid var(--line);border-radius:10px;min-height:120px}
table{border-collapse:separate;border-spacing:0;width:100%;font-variant-numeric:tabular-nums}
th,td{padding:7px 10px;text-align:right;white-space:nowrap;border-bottom:1px solid var(--line)}
th{position:sticky;top:0;background:var(--panel);z-index:2;font-size:12px;color:var(--muted);font-weight:600;cursor:pointer;user-select:none;border-bottom:2px solid var(--line)}
th:hover{color:var(--text)}
th.on{color:var(--accent)}
td.l,th.l{text-align:left}
tbody tr.row{cursor:pointer}
tbody tr.row:hover{background:var(--hover)}
tr.open td{background:var(--hover)}
td.name{font-weight:600}
.rk{color:var(--muted);width:42px}
.pos{color:var(--pos)}.neg{color:var(--neg)}
.badge{font-size:11px;padding:1px 7px;border-radius:99px;font-weight:600}
.badge.pos{background:var(--posbg)}.badge.neg{background:var(--negbg)}
mark{background:var(--mark);color:inherit;border-radius:2px}
.chart{position:relative;width:170px;height:16px;margin-left:auto}
.chart .ax{position:absolute;left:50%;top:0;bottom:0;width:1px;background:var(--muted);opacity:.5}
.chart .bar{position:absolute;top:3px;bottom:3px;border-radius:2px}
.chart .bar.p{background:var(--pos);opacity:.75}.chart .bar.n{background:var(--neg);opacity:.75}
.chart .ci{position:absolute;top:7px;height:2px;background:var(--text);opacity:.65}
tr.det>td{background:var(--bg);padding:10px 16px 14px;text-align:left;white-space:normal}
.det table{width:auto;min-width:560px;background:var(--panel);border:1px solid var(--line);border-radius:8px}
.det th{position:static;cursor:default}
.det .note{color:var(--muted);font-size:12px;margin-bottom:6px}
.empty{padding:40px;text-align:center;color:var(--muted)}
.kbd{font-size:11px;border:1px solid var(--line);border-radius:4px;padding:0 4px;color:var(--muted)}
</style></head>
<body>
<div class="app">
  <div>
    <h1>Which professors grade above or below their colleagues?</h1>
    <div class="sub" id="meta"></div>
  </div>

  <div class="toolbar">
    <div class="search">
      <svg viewBox="0 0 20 20" fill="none" stroke="currentColor" stroke-width="2"><circle cx="8.5" cy="8.5" r="5.5"/><path d="M13 13l4 4"/></svg>
      <input id="q" type="search" placeholder="Search name, subject or course (e.g. PHYS 111)…  press /" autocomplete="off">
      <button class="x" id="qx" title="Clear">×</button>
    </div>
    <select id="subj" title="Filter by main subject"></select>
    <div class="seg" id="dir">
      <button data-v="all" class="on">All</button><button data-v="above">Above</button><button data-v="below">Below</button>
    </div>
    <label class="chk"><input type="checkbox" id="clear"> Clear gaps only</label>
    <span class="lab">Min sections</span><input class="num" id="minsec" type="number" min="0" value="0">
    <select id="sort" title="Sort order"></select>
    <button class="btn" id="reset">Reset</button>
    <button class="btn" id="export">Export CSV</button>
    <button class="btn" id="help">How to read</button>
  </div>

  <div class="info" id="info">
    <b>How to read the table</b>
    <ul>
      <li><b>Diff</b> = the professor's average minus everyone else's average <i>in the same courses</i>, in percentage points, weighted by how many of the professor's students were in each course. Positive = their sections average higher than colleagues teaching those same courses.</li>
      <li><b>Prof avg / Others avg</b> use the same course mix and weights, so they subtract cleanly.</li>
      <li><b>Median course diff</b> and <b>% courses above</b> show whether the gap is consistent or driven by one course.</li>
      <li><b>95% CI</b> is a bootstrap interval for Diff (resampling courses). <b>Clear gap</b> = the interval excludes 0. In the chart, the bar is Diff and the thin line is the CI.</li>
      <li>Click any row to see the course-by-course breakdown.</li>
    </ul>
    <b>Caveats</b>
    <ul>
      <li>With hundreds of professors, a handful will show a "clear gap" by chance alone. Treat the extremes as leads, not verdicts.</li>
      <li>Averages also reflect class size, term, course mix, and which students enrol; they are not a measure of teaching quality.</li>
      <li>Co-taught sections count for each listed instructor. Older terms may also list TAs, which the thresholds mostly filter out.</li>
    </ul>
  </div>

  <div class="status" id="status"></div>
  <div class="tablebox"><table><thead id="thead"></thead><tbody id="tbody"></tbody></table></div>
</div>

<script>
const DATA = __DATA__;
const META = __META__;

const COLS = [
  {k:"_rank", label:"#", cls:"rk", nosort:true},
  {k:"Professor", label:"Professor", l:true},
  {k:"Main subjects", label:"Main subjects", l:true},
  {k:"Sections", label:"Sections"},
  {k:"Terms", label:"Terms"},
  {k:"Courses compared", label:"Courses"},
  {k:"Students", label:"Students"},
  {k:"Prof avg", label:"Prof avg", d:1},
  {k:"Others avg", label:"Others avg", d:1},
  {k:"Diff", label:"Diff", d:1},
  {k:"_chart", label:"Gap vs others", nosort:true},
  {k:"CI low", label:"95% CI", ci:true},
  {k:"Median course diff", label:"Median course diff", d:1},
  {k:"% courses above", label:"% courses above", d:0},
  {k:"Clear gap?", label:"Clear gap"}
];

const state = {q:"", subj:"", dir:"all", clear:false, minSec:0, sk:"Diff", desc:true, open:new Set()};
if (META.sort_by_abs) { state.sk = "abs"; }

const $ = s => document.querySelector(s);
const esc = s => String(s).replace(/[&<>"']/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
const fmt = (v, d) => (v === null || v === undefined || isNaN(v)) ? "" : Number(v).toFixed(d);
const sgn = v => v > 0 ? "+" : "";

// precompute search helpers
DATA.forEach(r => {
  r._subs = (r["Main subjects"] || "").split(",").map(s => s.trim()).filter(Boolean);
  r._hay = [r.Professor, r["Main subjects"], (r.courses || []).map(c => c.course).join(" ")].join(" ").toLowerCase();
});
const SCALE = Math.max(1, ...DATA.map(r => Math.max(Math.abs(r["CI low"]), Math.abs(r["CI high"]), Math.abs(r.Diff)))) * 1.05;
const X = v => 50 + (v / SCALE) * 50;

function hl(name, toks) {
  const ts = toks.filter(t => t.length);
  if (!ts.length) return esc(name);
  const re = new RegExp("(" + ts.map(t => t.replace(/[.*+?^${}()|[\]\\]/g, "\\$&")).join("|") + ")", "ig");
  return name.split(re).map((p, i) => i % 2 ? "<mark>" + esc(p) + "</mark>" : esc(p)).join("");
}

function getRows() {
  const toks = state.q.toLowerCase().split(/\s+/).filter(Boolean);
  let rows = DATA.filter(r => {
    if (toks.length && !toks.every(t => r._hay.includes(t))) return false;
    if (state.subj && !r._subs.includes(state.subj)) return false;
    if (state.dir === "above" && !(r.Diff > 0)) return false;
    if (state.dir === "below" && !(r.Diff < 0)) return false;
    if (state.clear && r["Clear gap?"] !== "yes") return false;
    if (r.Sections < state.minSec) return false;
    return true;
  });
  const val = r => state.sk === "abs" ? Math.abs(r.Diff) : r[state.sk];
  rows.sort((a, b) => {
    const x = val(a), y = val(b);
    const c = (typeof x === "string") ? x.localeCompare(y) : (x - y);
    return state.desc ? -c : c;
  });
  return {rows, toks};
}

function chart(r) {
  const x = X(r.Diff), lo = X(r["CI low"]), hi = X(r["CI high"]);
  const left = Math.min(50, x), w = Math.abs(x - 50);
  return `<div class="chart" title="Diff ${sgn(r.Diff)}${fmt(r.Diff,1)}  |  95% CI [${fmt(r["CI low"],1)}, ${fmt(r["CI high"],1)}]">
    <div class="ax"></div>
    <div class="bar ${r.Diff >= 0 ? "p" : "n"}" style="left:${left}%;width:${w}%"></div>
    <div class="ci" style="left:${lo}%;width:${Math.max(0, hi - lo)}%"></div></div>`;
}

function detail(r) {
  const cs = r.courses || [];
  let h = `<tr class="det"><td colspan="${COLS.length}"><div class="note"><b>${esc(r.Professor)}</b> vs everyone else teaching the same course (${cs.length} comparable courses, largest classes first)</div>
  <table><thead><tr><th class="l">Course</th><th>Their sections</th><th>Others' sections</th><th>Students</th><th>Their avg</th><th>Others avg</th><th>Diff</th></tr></thead><tbody>`;
  cs.forEach(c => {
    h += `<tr><td class="l">${esc(c.course)}</td><td>${c.sections}</td><td>${c.others_sections}</td><td>${c.students}</td>
      <td>${fmt(c.prof,1)}</td><td>${fmt(c.others,1)}</td><td class="${c.diff >= 0 ? "pos" : "neg"}">${sgn(c.diff)}${fmt(c.diff,1)}</td></tr>`;
  });
  return h + "</tbody></table></td></tr>";
}

function render() {
  const {rows, toks} = getRows();
  // header
  $("#thead").innerHTML = "<tr>" + COLS.map(c => {
    const on = (c.k === state.sk) || (c.ci && state.sk === "CI low");
    const arrow = on ? (state.desc ? " ▼" : " ▲") : "";
    return `<th class="${c.l ? "l " : ""}${on ? "on" : ""}" data-k="${c.k}" ${c.nosort ? 'style="cursor:default"' : ""}>${c.label}${arrow}</th>`;
  }).join("") + "</tr>";
  // body
  let h = "";
  rows.forEach((r, i) => {
    const open = state.open.has(r.Professor);
    h += `<tr class="row ${open ? "open" : ""}" data-p="${esc(r.Professor)}">` + COLS.map(c => {
      switch (c.k) {
        case "_rank": return `<td class="rk">${i + 1}</td>`;
        case "Professor": return `<td class="l name">${hl(r.Professor, toks)}</td>`;
        case "Main subjects": return `<td class="l">${esc(r["Main subjects"])}</td>`;
        case "Diff": return `<td class="${r.Diff >= 0 ? "pos" : "neg"}"><b>${sgn(r.Diff)}${fmt(r.Diff,1)}</b></td>`;
        case "_chart": return `<td>${chart(r)}</td>`;
        case "CI low": return `<td>[${fmt(r["CI low"],1)}, ${fmt(r["CI high"],1)}]</td>`;
        case "Clear gap?": return `<td>${r["Clear gap?"] === "yes" ? `<span class="badge ${r.Diff > 0 ? "pos" : "neg"}">${r.Diff > 0 ? "above" : "below"}</span>` : ""}</td>`;
        case "Median course diff": return `<td>${sgn(r[c.k])}${fmt(r[c.k], 1)}</td>`;
        default: return `<td>${c.d !== undefined ? fmt(r[c.k], c.d) : esc(r[c.k])}</td>`;
      }
    }).join("") + "</tr>";
    if (open) h += detail(r);
  });
  $("#tbody").innerHTML = h || `<tr><td colspan="${COLS.length}" class="empty">No professors match these filters.</td></tr>`;
  // status
  const above = rows.filter(r => r.Diff > 0).length, below = rows.filter(r => r.Diff < 0).length;
  const clr = rows.filter(r => r["Clear gap?"] === "yes").length;
  $("#status").innerHTML = `<span>Showing <b>${rows.length}</b> of ${DATA.length} professors</span><span><b>${above}</b> above others</span><span><b>${below}</b> below</span><span><b>${clr}</b> with a clear gap</span>`;
  // sort dropdown
  const key = state.sk + ":" + (state.desc ? "d" : "a");
  const sel = $("#sort"); sel.value = [...sel.options].some(o => o.value === key) ? key : "custom";
  $("#qx").style.display = state.q ? "block" : "none";
  [...$("#dir").children].forEach(b => b.classList.toggle("on", b.dataset.v === state.dir));
}

// ---- wire up controls ----
$("#meta").textContent = `${META.campus} · subjects: ${META.subjects} · taught ≥ ${META.min_sections} sections over ≥ ${META.min_terms} terms · ≥ ${META.min_courses} comparable courses (others taught each in ≥ ${META.min_other} sections) · built ${META.built}`;

const subs = [...new Set(DATA.flatMap(r => r._subs))].sort();
$("#subj").innerHTML = `<option value="">All subjects</option>` + subs.map(s => `<option>${esc(s)}</option>`).join("");
$("#sort").innerHTML = `<option value="Diff:d">Most above others</option><option value="Diff:a">Most below others</option>
  <option value="abs:d">Largest gap (either way)</option><option value="Professor:a">Name A–Z</option>
  <option value="Students:d">Most students</option><option value="custom" disabled>Custom (column header)</option>`;

let t;
$("#q").addEventListener("input", e => { clearTimeout(t); t = setTimeout(() => { state.q = e.target.value; render(); }, 80); });
$("#qx").onclick = () => { $("#q").value = ""; state.q = ""; render(); $("#q").focus(); };
$("#subj").onchange = e => { state.subj = e.target.value; render(); };
$("#dir").onclick = e => { if (e.target.dataset.v) { state.dir = e.target.dataset.v; render(); } };
$("#clear").onchange = e => { state.clear = e.target.checked; render(); };
$("#minsec").oninput = e => { state.minSec = Number(e.target.value) || 0; render(); };
$("#sort").onchange = e => { const [k, d] = e.target.value.split(":"); state.sk = k; state.desc = d === "d"; render(); };
$("#help").onclick = () => $("#info").classList.toggle("show");
$("#reset").onclick = () => {
  Object.assign(state, {q:"", subj:"", dir:"all", clear:false, minSec:0, sk: META.sort_by_abs ? "abs" : "Diff", desc:true});
  state.open.clear();
  $("#q").value = ""; $("#subj").value = ""; $("#clear").checked = false; $("#minsec").value = 0;
  render();
};
$("#thead").onclick = e => {
  const th = e.target.closest("th"); if (!th) return;
  const col = COLS.find(c => c.k === th.dataset.k); if (!col || col.nosort) return;
  const k = col.k;
  if (state.sk === k) state.desc = !state.desc; else { state.sk = k; state.desc = k !== "Professor" && k !== "Main subjects"; }
  render();
};
$("#tbody").onclick = e => {
  const tr = e.target.closest("tr.row"); if (!tr) return;
  const p = tr.dataset.p;
  state.open.has(p) ? state.open.delete(p) : state.open.add(p);
  render();
};
$("#export").onclick = () => {
  const {rows} = getRows();
  const keys = Object.keys(DATA[0]).filter(k => k[0] !== "_" && k !== "courses");
  const q = v => '"' + String(v).replace(/"/g, '""') + '"';
  const csv = [keys.map(q).join(",")].concat(rows.map(r => keys.map(k => q(r[k])).join(","))).join("\n");
  const a = document.createElement("a");
  a.href = URL.createObjectURL(new Blob([csv], {type: "text/csv"}));
  a.download = "prof_comparison_filtered.csv"; a.click();
};
document.addEventListener("keydown", e => {
  if (e.key === "/" && document.activeElement !== $("#q")) { e.preventDefault(); $("#q").focus(); }
  if (e.key === "Escape" && document.activeElement === $("#q")) { $("#qx").click(); }
});
render();
</script>
</body></html>
"""


def build_dashboard(df, details, meta):
    """Return one self-contained HTML page (no internet needed) for the results table."""
    recs = json.loads(df.reset_index(drop=True).to_json(orient="records"))
    for r in recs:
        r["courses"] = details.get(r["Professor"], [])
    # "<" is escaped so a stray "</script>" in a name can never break the page
    data = json.dumps(recs).replace("<", "\\u003c")
    return (DASHBOARD_TEMPLATE
            .replace("__DATA__", data)
            .replace("__META__", json.dumps(meta).replace("<", "\\u003c")))


def show_dashboard(page, height=820):
    """Show the page inline in Jupyter."""
    frame = (f'<iframe srcdoc="{_html.escape(page, quote=True)}" width="100%" height="{height}" '
             f'style="border:1px solid #8884;border-radius:10px"></iframe>')
    display(HTML(frame))


## Run it

In [ ]:
sections = load_data(CAMPUS, SUBJECTS, CSV_DIR, CACHE_FILE, REFRESH, DEBUG)

profs, n_sec_all, _ = select_profs(sections, MIN_SECTIONS, MIN_TERMS)
print(f"\n{len(n_sec_all)} instructor names seen; {len(profs)} have taught "
      f">= {MIN_SECTIONS} sections across >= {MIN_TERMS} terms.")

details = {}
df = compare_all(sections, profs, MIN_COURSES, MIN_OTHER_SECTIONS, details=details)
if df.empty:
    print("No professors qualified. Lower MIN_SECTIONS / MIN_TERMS / MIN_COURSES in the settings cell.")
else:
    key = df["Diff"].abs() if SORT_BY_ABS else df["Diff"]
    df = df.loc[key.sort_values(ascending=False).index].reset_index(drop=True)
    df.index += 1
    df.index.name = "Rank"
    print(f"{len(df)} professors have >= {MIN_COURSES} comparable courses. "
          f"Sorted by {'absolute ' if SORT_BY_ABS else ''}difference "
          f"({'largest gap first' if SORT_BY_ABS else 'most above others first'}).\n")

    df.round(2).to_csv(OUT_CSV)
    print(f"Saved CSV to {os.path.abspath(OUT_CSV)}")

    meta = {"campus": CAMPUS, "subjects": SUBJECTS, "min_sections": MIN_SECTIONS, "min_terms": MIN_TERMS,
            "min_courses": MIN_COURSES, "min_other": MIN_OTHER_SECTIONS, "sort_by_abs": bool(SORT_BY_ABS),
            "built": date.today().isoformat()}
    page = build_dashboard(df.round(2), details, meta)
    with open(OUT_HTML, "w", encoding="utf-8") as fh:
        fh.write(page)
    print(f"Saved dashboard to {os.path.abspath(OUT_HTML)} (open it in a browser for full screen)\n")

    show_dashboard(page, DASH_HEIGHT)


### How to read the table
- **Diff** = the professor's average minus everyone else's average *in the same courses*, in percentage points, weighted by how many of the professor's students were in each course. Positive means their sections average higher than colleagues teaching those same courses.
- **Prof avg / Others avg** use the same course mix and weights, so they subtract cleanly.
- **Median course diff** and **% courses above** show whether the gap is consistent or driven by one course.
- **CI low / CI high** is a 95% bootstrap interval for Diff (resampling courses). **Clear gap?** = yes when the interval excludes 0.

### Caveats
- With a hundred or more professors, a handful will show a "clear gap" by chance alone. Treat the extremes as leads, not verdicts.
- Averages also reflect class size, term, course mix, and which students enrol; they are not a measure of teaching quality.
- Sections list every named instructor, and co-taught sections count for each listed person. Older terms may also list TAs, which the thresholds mostly filter out.